# Julia notebook to *use* the symbolic solution to the frictional geostrophic equations with a specified linear relaxation buoyancy field and surface wind stress.

twnh Dec '25

This notebook uses the Green's function solution to:

\begin{align}
- f  v & = - \frac{\partial p}{\partial x} + \epsilon^2   \nu \frac{d^2 u}{d z^2}, \\
  f  u & = - \frac{\partial p}{\partial y} + \epsilon^2   \nu \frac{d^2 v}{d z^2} , 
\end{align}
for $u(z), v(z)$, given viscosity $\nu = \nu_0$, and boundary conditions
\begin{align}
\text{Surface~}z = 0:
\begin{cases}
\displaystyle \epsilon^2 \nu \partial_z u & = \tau^x \\
\displaystyle  \epsilon^2 \nu \partial_z v & = \tau^y 
\end{cases}
 \\
\text{Bottom~}z = -H:
\begin{cases}
u & = 0  \\
v & = 0 
\end{cases} ,
\end{align}
where $(\tau^x, \tau^y)$ is the surface wind stress, $\nu$ is the fixed viscosity, and $\epsilon$ is the aspect ratio.
The pressure $p$ is due to the surface pressure field $p_s(x,y)$ and buoyancy field $b(x,y,z)$:

\begin{align}
p(x,y,z) & = p_s(x,y) + \int_{z}^{0} b(x,y,z') \; dz' , \\
\implies p_b(x, y) & \equiv p(x, y, z=-H) = p_s(x,y) + \int_{-H(x,y)}^{0} b(x,y,z') \; dz' .
\end{align}

This code builds the equation satisfied by the surface pressure field $p_s(x,y)$ using the Green's function solution.
It then solves for the pressure field given a specified windstress and buoyancy forcing.

The buoyancy forcing comes from solving

\begin{align}
 \epsilon^2   \kappa \frac{d^2 b}{d z^2} + \gamma \left( B(z) - b \right) = 0
\end{align}
for $b(z)$, given fixed diffusivity $\kappa$, and boundary conditions
\begin{align}
\text{Surface~}z = 0: b &= 0 ,\\
\text{Bottom~}z = -H: \frac{\partial b}{\partial z} &= 0
\end{align}
where $B(z)$ is the specified relaxation buoyancy field and $\gamma$ is the relaxation rate. Again, this problem is solved using a Green's function that is specified in this notebook.

This version extends `ExampleSolution_v_0.7.ipynb` to utilize a new double basin bathymetry.

In [ ]:
notebook_name = "ExampleSolution_v0.8"
using Symbolics
@variables I        # Placeholder variable for imaginary unit, otherwise Symbolics.jl computes things like sqrt(im) as a numerical value.
using Gridap
using GridapGmsh
using WriteVTK

### Define symbols:

In [ ]:
# Geometry symbolic parameters:
@variables z::Real ξ::Real      # Both (z, ξ) in [-H, 0]
@variables x::Real y::Real
@variables H(x,y)::Real         # Function H(x, y), depth

# Frictional geostrophic equation symbolic parameters:
@variables f::Real ν::Real ϕ::Real ϵ::Real
root_iϕ = I^(1//2) * ϕ
@variables τs(x,y)     # Complex surface wind stress τs(x, y)
@variables psg(x,y)    # Complex surface pressure gradient (∂/∂x + i ∂/∂y) p_s(x, y)
@variables b(x,y,z)::Real       # Buoyancy field b(x, y, z)

# Buoyancy equation symbolic parameters:
@variables κ::Real ψ::Real γ::Real α::Real
@variables Bξ(x,y,ξ)::Real

# Differential operators:
∂x = Differential(x)
∂y = Differential(y)

#### Function to build callable function from `Symbolics` expression. This is where `I` gets replaced with $\sqrt{-1}$:

In [ ]:
function build_Symbolics_fn(expr,varnames)
	vars = [Symbolics.Variable(Symbol(n)) for n in varnames]							# Extract Symbolics variables from variable names
	I_vars = filter(v -> string(v) == "I", Symbolics.get_variables(expr))				# Check if imaginary unit variable is present and substitute
	if !isempty(I_vars)
    	tmp1 = substitute(expr, Dict(first(I_vars) => im))
	else
	    tmp1 = expr
	end
	tmp2 = expand_derivatives(tmp1)														# Expand derivatives
	tmp3vec = build_function(tmp2, vars; expression=Val{false})
	
	# Wrap function to throw an error if it's called with multiple scalar arguments rather than a vector:
	function tmp3(args::AbstractVector)
    	length(args) == length(vars) || throw(ArgumentError("Input must be vector of length $(length(vars))."))
    	tmp3vec(args)
	end
	tmp3(args...) = throw(MethodError(tmp3, args))
	return tmp3
end

### Define parameter values:

In [ ]:
# Define the parameter values
f_val = 1.0
@info "Notebook is hard-coded for constant Coriolis parameter."
ϵ_val = 0.95
ν_val = 0.06
κ_val = ν_val
γ_val = 0.2
α_val = 0.3
Ekman_depth = sqrt(2*ν_val/f_val)

# Compute compound parameter:
ϕ_val = sqrt(f_val / ν_val) / ϵ_val
ψ_val = sqrt(γ_val / κ_val) / ϵ_val

# Define domain depth, and hence the geometry and mesh:
GmshMeshFileName = "reservoir.msh"
include("build_mesh.jl")
# H_val(x,y)    = 1.0 - x^2 - y^2 + (1.0 - (x-xc)^2 - y^2)
# H_val(x,y)    = (1.0 - ((x-(xc/2))/((xc+2)/2))^2 - y^2); xc = 2.5
# H_val(x,y)    = 1.0 - x^2 - y^2 ; xc = 0.0
hp = 1.0; hm = 0.1; xc = 1.0; r = 1.0; H_params = define_H_parameters(xc,r, hp, hm)
H_val(x,y)    = compute_H(x, y, H_params)
res = 0.05
build_geometry_and_mesh(H_val, xc, r, res, GmshMeshFileName)

# Define wind stress:
# τs_val(x,y) = 0.1 + 0.0im
function τs_val(x, y)
    r = sqrt(x^2 + y^2)
    V = 0.0 * r^2 
    return -V * y / r + im* V * x / r
end

# Define relaxation buoyancy field:
B_val(x,y,ξ)  = α * ξ       # Results below are hard-coded for this linear profile.
@info "Notebook is hard-coded for linear relaxation buoyancy profile."

# Define dictionaries for substitutions:
param_values = Dict(f=>f_val, ϵ=>ϵ_val, ν=>ν_val, ϕ=>ϕ_val, γ=>γ_val, α=>α_val, κ=>κ_val, ψ=>ψ_val)
fn_values    = Dict(τs=>τs_val(x,y), H=>H_val(x,y), Bξ=>B_val(x,y,ξ))

# Report
println()
println("This case parameter values:")
display(param_values)

println()
println("This case function values:")
display(fn_values)

println()
println("Non-dimensional Ekman_depth = $(Ekman_depth)")

### Compute the velocity G's function and various special cases of it:

In [ ]:
function define_Guv()                   # Taken from Latex notes and also from ExampleSolution_v0.5.ipynb on 3Dec25.
    prefactor = I^(3//2) * exp(-root_iϕ * (z + ξ)) / (2 * ν * ϵ^2 * root_iϕ * (1 + exp(2 * root_iϕ * H)))
    Guv = ifelse(z <= ξ,
          - exp(2 * root_iϕ * ξ) + exp(2 * root_iϕ * (z + H)) + exp(2 * root_iϕ * (z + ξ + H)) - 1,
          - exp(2 * root_iϕ * z) + exp(2 * root_iϕ * (ξ + H)) + exp(2 * root_iϕ * (z + ξ + H)) - 1)
    Guv = prefactor * Guv
    return Guv
end

function define_Guv_int_wrt_ξ()         # Taken from Latex notes and also from ExampleSolution_v0.5.ipynb on 3Dec25.
    Guv_int_wrt_ξ = - I * exp(- root_iϕ * z) * (exp(root_iϕ * z) - exp(root_iϕ * H)) * (exp(root_iϕ * (z + H)) - 1) / (ν * ϵ^2 * ϕ^2 * (1 + exp(2 * root_iϕ * H)))
    return Guv_int_wrt_ξ
end

Guv = define_Guv()
Guv0 = simplify(substitute(Guv,Dict(ξ => 0, z => -1.0 )))         # Compute Guv at ξ = 0, z = -1.0 (< ξ)
Guv_int_wrt_ξ = define_Guv_int_wrt_ξ()

### Compute the buoyancy field function:

In [ ]:
Latex_b = ((- α * γ)) / (κ * ψ^2 * ϵ^2) * ( 
(exp(ψ * H) * (exp(ψ * z) - exp(-ψ * z))) /(ψ * (1 + exp(2 * ψ * H)))
- z
)

In [ ]:
function define_S_fn()           # Taken from Latex notes and also from ExampleSolution_v0.5.ipynb on 3Dec25.
    S = (1/(root_iϕ))*tanh(root_iϕ * H)
    return S
end

function define_T_fn()           # Taken from Latex notes and also from ExampleSolution_v0.5.ipynb on 13Dec25.

# Construct the buoyancy related T(x) term (excludes the stress-driven part):
    Latex_Txb_term = α * γ * (exp((ψ + root_iϕ) * H) - exp((3*ψ + root_iϕ) * H))/
    (κ * ϵ^2 * ψ^3 * (1 + exp(2 * ψ * H))^2 * (1 + exp(2 * root_iϕ * H) ))
    Latex_Txb_term = Latex_Txb_term * (∂x(H) + I*∂y(H))
    Latex_int_result = (
        ((1 - exp(- (ψ - root_iϕ)*H))/( ψ - root_iϕ)) +
        ((1 - exp(- (ψ + root_iϕ)*H))/( ψ + root_iϕ)) +
        (2/root_iϕ) * (exp(-root_iϕ * H) - exp(root_iϕ * H)) +
        ((1 - exp(  (ψ + root_iϕ)*H))/(-ψ - root_iϕ)) + 
        ((1 - exp(  (ψ - root_iϕ)*H))/(-ψ + root_iϕ)) 
        )
    Latex_Txb_term = Latex_Txb_term * Latex_int_result
    # Add the wind-stress driven part:
    T = Latex_Txb_term - (2 * exp(root_iϕ * H)/(1 + exp(2 * root_iϕ*H))) * τs
    return T
end

function define_A_fn()           # Taken from Latex notes and also from ExampleSolution_v0.5.ipynb on 3Dec25.
    A = (I/f) * (H + (1 - exp(2 * root_iϕ * H))/(root_iϕ * (1 + exp(2 * root_iϕ * H))))
    return A
end

function define_B_fn()           # Taken from Latex notes and also from ExampleSolution_v0.5.ipynb on 13Dec25.
   
    # Construct the buoyancy related B(x) term (excludes the stress-driven part):
    Latex_Bxb_term = α * γ * (exp(ψ * H) - exp(3 * ψ * H))/(κ * ϵ^2 * ψ^3 * (1 + exp(2 * ψ * H))^2)
    Latex_Bxb_term = Latex_Bxb_term * (∂x(H) + I*∂y(H))
    Latex_int_result = exp(root_iϕ * H) * (
        ((1 - exp(- (ψ - root_iϕ)*H))/( ψ - root_iϕ)) +
        ((1 - exp(- (ψ + root_iϕ)*H))/( ψ + root_iϕ)) +
        (2/root_iϕ) * (exp(-root_iϕ * H) - exp(root_iϕ * H)) +
        ((1 - exp(  (ψ + root_iϕ)*H))/(-ψ - root_iϕ)) + 
        ((1 - exp(  (ψ - root_iϕ)*H))/(-ψ + root_iϕ)) +
        ((exp((-ψ + root_iϕ) * H) - exp((ψ + root_iϕ) * H))/ψ)
        )
    Latex_int_result = Latex_int_result + (1/ψ)*(exp(-ψ * H) - exp(ψ * H)) + 2*H*(1 + exp(2*root_iϕ * H))
    Latex_Bxb_term = Latex_Bxb_term * Latex_int_result 
    # Add the wind-stress driven part:
    B = Latex_Bxb_term + (exp(root_iϕ * H) - 1)^2 * τs
    # Final overall scaling:
    B = -I * B / (f * (1 + exp(2 * root_iϕ * H)))
    return B
end

### Solve for surface pressure using Gridap

In [ ]:
println()

@time "Solving for the surface pressure using Gridap..." begin

# 1. Define the mesh
    model = GmshDiscreteModel(GmshMeshFileName)
    Ω = Gridap.Geometry.Triangulation(model)
    dΩ = Measure(Ω, 2)

# 2. Define the finite element space (piecewise linear, Dirichlet zero BC)
    order = 2
    reffe_scalar = ReferenceFE(lagrangian, Float64, order)
    V     = TestFESpace(model, reffe_scalar; conformity=:H1, dirichlet_tags="boundary")
    U     = TrialFESpace(V)
    reffe_vector = ReferenceFE(lagrangian, VectorValue{2,Float64}, order)
    V_vec = TestFESpace(model, reffe_vector; conformity=:H1, dirichlet_tags="boundary")

# 3. Define the coefficients of the elliptic equation:
    A_fn_tmp0 = substitute(substitute(define_A_fn(),param_values),fn_values)
    A_fn_tmp = build_Symbolics_fn(A_fn_tmp0, [x, y])
    A_fn(xx) = (typeof(xx[1]) <: Real ? A_fn_tmp([xx[1],xx[2]]) : 1.0)     # Might get called with non-Float argument

    B_fn_tmp0 = substitute(substitute(define_B_fn(),param_values),fn_values)
    B_fn_tmp = build_Symbolics_fn(B_fn_tmp0, [x, y])
    B_fn(xx) = (typeof(xx[1]) <: Real ? B_fn_tmp([xx[1],xx[2]]) : 0.0)     # Might get called with non-Float argument

# 4. Define weak form (variational formulation)
    a(u,v) = ∫( real( (∇(v) ⋅ VectorValue( 1.0, -1im)) * (A_fn * (∇(u) ⋅ VectorValue(1.0, 1im))) ) )dΩ
    l(v)   = ∫( real( (∇(v) ⋅ VectorValue(-1.0,  1im)) *  B_fn ) )dΩ

# 5. Assemble and solve
    op = AffineFEOperator(a, l, U, V)
    psurf = Gridap.solve(op)  # This is your numerical solution as a Gridap FEFunction
end

#### Solve for streamfunction of depth-integrated flow:

Fast assembly of complex-valued function using cached intermediate functions and function composition, not function calls inside the integral.
Need to interpolate real and imaginary parts separately (Gridap does not support complex-valued FE functions directly).

See: [this Gridap gitter thread](https://matrix.to/#/!mSZoaZwNZhWulNruaK:gitter.im/$EWrGYx6lcBOQt05k5y2EGXangkwEitmxwFjaslhlSt8?via=gitter.im&via=matrix.org&via=mozilla.org)

In [ ]:
println()
@time "Computing streamfunction for vertically-integrated horizontal velocity field using Gridap..." begin
    reffe_vector = ReferenceFE(lagrangian, VectorValue{2,Float64}, order)
    V_vec = TestFESpace(model, reffe_vector; conformity=:H1, dirichlet_tags="boundary")
    gradpsurf_projection = interpolate(∇(psurf), V_vec)

    ReA_fe = interpolate(x -> real(A_fn(x)), U)
    ImA_fe = interpolate(x -> imag(A_fn(x)), U)
    ReB_fe = interpolate(x -> real(B_fn(x)), U)
    ImB_fe = interpolate(x -> imag(B_fn(x)), U)
    AU_fe = (ReA_fe + 1im*ImA_fe)
    BU_fe = (ReB_fe + 1im*ImB_fe)
    dot_gradp = gradpsurf_projection ⋅ VectorValue(1, 1im)
    C_fn = AU_fe * dot_gradp + BU_fe

    a2(u,v) = ∫( ∇(v) ⋅ ∇(u))dΩ
    l2(v)   = ∫( - real( (∇(v) ⋅ VectorValue(1im, 1.0)) *  C_fn ) )dΩ
    op2 = AffineFEOperator(a2, l2, U, V)
    Psi = Gridap.solve(op2)
    writevtk(Ω,notebook_name * "_solution.vtu",cellfields=["psurf"=>psurf, "Psi"=>Psi])
end

#### Setup helper functions:

In [ ]:
function ps_val(xx, yy)
    try
    	gp = evaluate(psurf, Gridap.Point(xx, yy))
	    return gp
    catch
        return 0.0
    end
end

function psg_val(xx, yy)
    try
	    tmp = evaluate(∇(psurf), Gridap.Point(xx, yy))
	    return tmp[1] + 1im*tmp[2] 
    catch
        return 0.0 + 1im*0.0
    end
end

function UV_val(xx,yy) 
    try
        psg_tmp = evaluate(∇(psurf), Gridap.Point(xx, yy))
        UV = A_fn_tmp([xx,yy])*(psg_tmp[1] + 1im*psg_tmp[2]) + B_fn_tmp([xx,yy])
        return UV
    catch
        return 0.0 + 1im*0.0
    end
end

function Psi_val(xx,yy)
    try
        Psi_tmp = evaluate(Psi, Gridap.Point(xx, yy))
        return Psi_tmp
    catch
        return 0.0
    end
end

### Solve for velocity field using the Green's function:

In [ ]:
println()

Guv0_fn		     = build_Symbolics_fn(substitute(substitute(Guv0,		   param_values),fn_values), [x, y, z])
Guv_int_wrt_ξ_fn = build_Symbolics_fn(substitute(substitute(Guv_int_wrt_ξ, param_values),fn_values), [x, y, z])
b_fn             = build_Symbolics_fn(substitute(substitute(Latex_b,       param_values),fn_values), [x, y, z])
S_fn_tmp		 = build_Symbolics_fn(substitute(substitute(define_S_fn(), param_values),fn_values), [x, y])
T_fn_tmp		 = build_Symbolics_fn(substitute(substitute(define_T_fn(), param_values),fn_values), [x, y])
τb_fn(xx,yy)     = S_fn_tmp([xx,yy])*psg_val(xx,yy) + T_fn_tmp([xx,yy])

# From Latex notes on 15Dec15:
Guv_exp_psi_xi_int_wrt_ξ = (1/(ν * ϵ^2 * (ψ^2 - I * ϕ^2))) * (
    ( I * sqrt(I) * ψ * exp(      ψ * H) * (exp(2 * root_iϕ * (z + H)) - 1) -
                    ϕ * exp(root_iϕ * H) * (exp(2 * root_iϕ *  z     ) + 1)
    ) * exp(-root_iϕ * z - ψ * H)
    /(ϕ * (1 + exp(2 * root_iϕ * H)))
    + exp(ψ * z)
	)
pbarog_term = ((α * γ)/(κ * ψ^3 * ϵ^2)) * ((exp(ψ * H) - exp(3 * ψ * H)) / (exp(2 * ψ * H) + 1)^2) * 
	(∂x(H) + I*∂y(H)) * 
	(Guv_exp_psi_xi_int_wrt_ξ - 2 * Guv_int_wrt_ξ + substitute(Guv_exp_psi_xi_int_wrt_ξ,Dict(ψ => -ψ)))
pbarog_term_fn = build_Symbolics_fn(substitute(substitute(pbarog_term,param_values),fn_values), [x, y, z])

Nx, Ny, Nz = 256, 129, 32
xs     = range(-1, xc+1, Nx)
ys     = range(-1,    1, Ny)
zs     = range(-1, 0, Nz)
us     = zeros(Nx, Ny, Nz)
vs     = zeros(Nx, Ny, Nz)
bs     = zeros(Nx, Ny, Nz)
Us     = zeros(Nx, Ny)
Vs     = zeros(Nx, Ny)
τxs    = zeros(Nx, Ny)
τys    = zeros(Nx, Ny)
τbxs   = zeros(Nx, Ny)
τbys   = zeros(Nx, Ny)
pss    = zeros(Nx, Ny)
spds   = zeros(Nx, Ny)
Psis   = zeros(Nx, Ny)
Hs     = zeros(Nx, Ny)

@time "Computing velocity field from surface pressure, Green's function, and known parameter and fields:" begin
for (ix, xx) in enumerate(xs)
	for (iy, yy) in enumerate(ys)
		this_H = H_val(xx, yy)
		if this_H >= 0
			this_pss_val = ps_val( xx, yy)      # Interpolate or evaluate ps
			this_psg_val = psg_val(xx, yy)      # Interpolate or evaluate psg
			this_τs_val  = τs_val(xx, yy)		# Compute surface stress
			# if iy == ceil(Int,Ny/2)				# Only compute 2D slice at y=0: for axisymmetric cases
			if iy < ceil(Int,Ny/2)				# Only compute half of domain in y: for symmetric cases
				for (iz, zz) in enumerate(zs)
					if zz <= 0 && zz >= -this_H 
						the_psg_term	= Guv_int_wrt_ξ_fn([xx, yy, zz]) * this_psg_val 
						the_pbarog_term = pbarog_term_fn(  [xx, yy, zz])
						the_stress_term = Guv0_fn(         [xx, yy, zz]) * this_τs_val
						this_uv_value = the_psg_term + the_pbarog_term + the_stress_term
						us[ix, iy, iz] = real(this_uv_value)
						vs[ix, iy, iz] = imag(this_uv_value)
						bs[ix, iy, iz] = b_fn([xx, yy, zz])
					end
				end
			end
			# Compute 2D fields here:
			UV             = UV_val(xx,yy)
			Us[    ix, iy] = float(real(UV))
			Vs[    ix, iy] = float(imag(UV))
			spds[  ix, iy] = sqrt(Us[ix, iy]^2 + Vs[ix, iy]^2)
			τxs[   ix, iy] = float(real(this_τs_val))
			τys[   ix, iy] = float(imag(this_τs_val))
			τb_val         = τb_fn(xx,yy)
			τbxs[  ix, iy] = float(real(τb_val))
			τbys[  ix, iy] = float(imag(τb_val))
			pss[   ix, iy] = this_pss_val
			Psis[  ix, iy] = Psi_val(xx,yy)
			Hs[    ix, iy] = this_H
		end
	end
end
end

# Write out solution for display by Paraview:
vtk_grid(notebook_name * "_3D_solution.vti", xs, ys, zs) do vtk
	vtk["u_speed"]  = us
	vtk["v_speed"]  = vs
	vtk["b_field"]  = bs
end

vtk_grid(notebook_name * "_2D_solution.vti", xs, ys) do vtk
	vtk["U_speed"]      = Us
	vtk["V_speed"]      = Vs
	vtk["speed"]        = spds
	vtk["x_sfc_stress"] = τxs
	vtk["y_sfc_stress"] = τys
	vtk["x_bot_stress"] = τbxs
	vtk["y_bot_stress"] = τbys
	vtk["sfc_p"]        = pss
	vtk["Psi"]          = Psis
	vtk["Depth"]        = Hs
end